# Show a dataset in any notebook (JupyterLab, VS Code, Colab)

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/python_package/f_notebooks_tutorials/41_show_a_dataset_in_any_notebook.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

One line shows an AnnData, or any file Cellucid reads, in the Cellucid
viewer inside the notebook cell's output. The same line works in
JupyterLab, Jupyter Notebook 7, VS Code, Google Colab, marimo and
Databricks, on your laptop or on a remote server: the viewer is a
notebook widget that reaches the kernel through the notebook's own
connection, so there is no port, proxy or server address to set up.

**Time:** 5 minutes. **You need:** `pip install cellucid` (it installs
`anywidget`), and a browser that can load https://www.cellucid.com (the
app; your data does not go there).

The same tutorial in R: [View an object in a notebook](../../r_package/f_notebooks_tutorials/07_view_in_a_notebook.ipynb).

In [ ]:
# A fresh runtime (Google Colab, a new environment) installs Cellucid once.
import importlib.util

if importlib.util.find_spec("cellucid") is None:
    %pip install --quiet cellucid

## The data

The 2,531-cell mouse pancreas (endocrinogenesis, day 15.5) AnnData checked
into the cellucid-python repository: UMAP in `obsm['X_umap_2d']`, cell types
in `obs['clusters']`, 27,998 genes. In a clone of the repository the cell
below finds it; anywhere else (Colab) it downloads it once. Use your own
`adata` in its place.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad

import cellucid

NAME = "endocrinogenesis_day15.5_raw.h5ad"
URL = (
    "https://raw.githubusercontent.com/theislab/cellucid-python/main/"
    "docs/user_guide/python_package/f_notebooks_tutorials/datasets/" + NAME
)


def pancreas_h5ad() -> Path:
    """The Pancreas file of a cellucid-python clone, else a copy downloaded once (35 MB)."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for candidate in (base / "datasets" / NAME, base / "docs/user_guide/python_package/f_notebooks_tutorials/datasets" / NAME):
            if candidate.is_file():
                return candidate
    if not (here / NAME).is_file():
        urlretrieve(URL, here / NAME)
    return here / NAME


DATA_PATH = pancreas_h5ad()
adata = ad.read_h5ad(DATA_PATH)
adata

## The one line

`show_anndata()` takes an `AnnData`, a `MuData`, or the path of a file
or folder Cellucid reads (`.h5ad`, Zarr, `.h5mu`, `.loom`, 10x,
MatrixMarket, `.rds`). A path is read in place, only what a view
needs; an in-memory `AnnData` is served in priority order, made from it as
the viewer reads it.
`dataset_name` and `dataset_id` name the dataset (a file written by
`cellucid.prepare()` names itself, so it needs neither).

In [ ]:
DATASET_ID = "pancreas-e15-5"

viewer = cellucid.show_anndata(DATA_PATH, dataset_name="Pancreas E15.5", dataset_id=DATASET_ID)

The viewer appears below the cell once the cell finishes.

| Notebook | What to know |
|---|---|
| JupyterLab, Jupyter Notebook 7 | `anywidget` must be installed where the notebook *server* runs (it usually is the same environment). JupyterLab is tested by the project. |
| VS Code | The Jupyter extension shows widgets; nothing else to set up. |
| Google Colab | Nothing to set up: anywidget enables Colab's widget manager. |
| marimo | Make the widget the cell's output: end the cell with `viewer`. |
| Databricks | Shown at once, as in Jupyter. |

Only JupyterLab is tested end to end by the project; the others show
anywidgets with the same protocol. If one shows nothing,
`viewer.debug_connection()` (below) says how far it got. The full
matrix: [Supported environments](../e_jupyter_hooks/03_supported_environments_matrix.md).

## Drive the viewer from Python

Commands go out at once; one sent before the viewer has loaded waits
and is delivered when it connects. Cells are named by obs name (or by
row).

In [ ]:
viewer.set_color_by("clusters")
beta = adata.obs_names[adata.obs["clusters"] == "Beta"]
viewer.highlight_cells(beta, color="#E76F51", label="Beta (from Python)")

## Between cells

A notebook hands a widget's messages to the kernel **between cells**.
So the viewer loads once the cell that showed it finishes, hooks run
while the kernel is idle (when you are using the viewer), and a call
that would wait for the viewer's answer in the same cell
(`wait_for_ready()`, a session request's `result()`) raises at once and says
what to do instead. Everything the viewer makes is already in the
kernel by the next cell.

Now draw a lasso in the viewer: in **Highlighting**, choose **Lasso**,
drag around some cells and click **Confirm**. Then run the next cell.

In [ ]:
selection = viewer.selection  # None until you confirm a selection
if selection is not None:
    print(len(selection), "cells selected with the", selection.source)
    print(adata.obs.loc[selection.obs_names, "clusters"].value_counts()[lambda counts: counts > 0])

Or react to every selection as it is made:

In [ ]:
@viewer.on_selection
def summarize(event):
    cells = event["cells"]
    counts = adata.obs["clusters"].iloc[cells].value_counts()
    print(f"{len(cells)} cells:", counts[counts > 0].to_dict())

## If nothing shows

`debug_connection()` waits for nothing, so it works in any cell. It
says where the app is loaded from, which outputs are connected,
whether the viewer's channel is open and what the viewer last sent.

In [ ]:
report = viewer.debug_connection()
{key: report[key] for key in ("transport", "web_source_url", "connected_outputs", "channel_connected")}

- `connected_outputs` empty: the notebook did not draw the widget.
  In JupyterLab or Notebook 7, install `anywidget` in the notebook
  server's environment and reload the page.
- `channel_connected` false with an output: the app at
  `web_source_url` did not load (a network that blocks
  www.cellucid.com: host a copy and pass `web_source_url=`).

A viewer in a **browser tab** of its own instead (a plain Python
script, a cluster through an SSH tunnel) is
`cellucid.AnnDataViewer(...)`, `cellucid.serve_anndata(...)` or
`cellucid serve data.h5ad`: the same API, served from this machine.

## Stop

`stop()` freezes the output on its last view and releases what it
served for an in-memory object.

In [ ]:
viewer.stop()

## Next

- [Pull everything back](42_pull_everything_back.ipynb): highlights,
  labels and scores onto `adata.obs` in one call.
- [Push clusters, refine, pull back](43_push_clusters_refine_and_pull.ipynb).
- R: [View an object in a notebook](../../r_package/f_notebooks_tutorials/07_view_in_a_notebook.ipynb).